In [0]:
%sql

SELECT 'Products' AS TableName, COUNT(*) AS Records FROM Products UNION ALL
SELECT 'Customers', COUNT(*) FROM Customers UNION ALL SELECT 'Sales',
COUNT(*) FROM Sales UNION ALL SELECT 'Customer Feedback', COUNT(*) FROM
customer_feedback;

--Part 1: Basic SQL Queries (Beginner)
--O1.1 Product Catalog
SELECT ProductID, ProductName, UnitPrice 
FROM products
ORDER BY UnitPrice desc;


--O1.2 Customer Count
select Region, count(CustomerID) as Number_of_Customers 
from customers
Group by Region;


--Q1.3 Recent Orders
select OrderID, OrderDate, TotalSales
from sales
order by OrderDate desc
limit 10;


--Q1.4 Affordable Products
select ProductName, Category, UnitPrice
from products
where UnitPrice < 1000;


--Q1.5 Customer Satisfaction Summary
select Satisfaction, count(satisfaction) as Number_of_responses_per_satisfaction_level
from customer_feedback
group by Satisfaction
order by number_of_responses_per_satisfaction_level desc;

-------------------------------------------------------------

-- 5. Part 2: Intermediate SQL Queries

--Q2.1 Sales by Category
select products.Category, round(sum(sales.totalsales),2) as Total_sales_per_Category, round(sum(sales.profit),2) as Total_profit_per_Category
from sales
inner join products
on sales.ProductID = products.ProductID
group by products.Category
order by total_sales_per_category desc;


--Q2.2 Top Customers
select  concat(firstname," ", lastname) as Customer_Fullname, round(sum(sales.totalsales),2) as Total_sales_per_customer
from sales 
inner join customers 
on sales.CustomerID = customers.CustomerID 
group by customer_fullname
order by total_sales_per_customer desc
limit 5;
 

--Q2.3 Monthly Sales Trend
select month(orderdate) as Month_of_2024, round(sum(totalsales),2) as Total_sales_per_month
from sales
group by month_of_2024;


--Q2.4 Channel Performance
select Channel, count(orderid) as Number_of_Orders, round(avg(totalsales),2) as Average_Order_value, round(sum(totalsales),2) as Total_Sales_Channel
from sales
group by channel;


--Q2.5 Product Performance with Ratings
select ProductCategory, round(avg(rating),2) as Average_Rating_per_Category, count(Satisfaction) as Number_of_Reviews 
from customer_feedback 
group by ProductCategory
having Number_of_Reviews >50;

--------------------------------------------------------
--Part 3: Advanced SQL Queries

--Q3.1 Best Selling Product per Category
SELECT ProductID, ProductName, Product_category, Total_sales_of_that_Product
FROM (
  SELECT 
          products.ProductID, 
          products.productname,
         MAX(products.category) as Product_category, 
         round(SUM(sales.totalsales),2) as Total_sales_of_that_Product,
         ROW_NUMBER() OVER (PARTITION BY MAX(products.category) ORDER BY round(SUM(sales.totalsales),2) DESC) Row_Num
  FROM Products  
  INNER JOIN sales 
  on products.ProductID = sales.productID
  GROUP BY products.ProductID, products.productname
) 
WHERE Row_Num <= 1;


--Q3.2 Customer Lifetime Value
Select CustomerID, Region, Channel , Customer_Total_Purchases, Number_of_Orders, Average_Order_Value
from
(
  select 
   customers.customerID,
   customers.region Region,
   sales.channel,
   ROUND(SUM(Sales.totalsales),2) Customer_Total_Purchases,
   COUNT(sales.OrderID) Number_of_Orders,
   ROUND(AVG(sales.totalsales),2) Average_Order_Value

   from sales
   inner join customers
   on customers.customerID = sales.CustomerID
   group by customers.customerID, customers.Region, sales.Channel
   
) 
group by CustomerID, Region, Channel , Customer_Total_Purchases, Number_of_Orders, Average_Order_Value
having number_of_orders >= 3 
order by Customer_Total_Purchases DESC
limit 10;

--Q3.3 Profit Margin Analysis
SELECT ProductName, Category, Total_sales, Total_Profit, Profit_Margin_Percent
FROM 
(     
 select 
   products.productname,
   products.category,
   ROUND(SUM(sales.totalsales),2) as Total_sales,
   ROUND(SUM(sales.profit),2) as Total_Profit, 
   ROUND(100* Total_Profit/SUM(sales.totalsales), 2) as Profit_Margin_Percent
   --SUM(totalsales) over(partition by category order by SUM(profit)) as Total_sales  
   from sales 
   inner join Products
   on Products.productid = sales.Productid
   group by products.productname, products.category
   
)

order by profit_margin_percent DESC
LIMIT 10;

--Q3.4 Year-over-Year Growth
--cte that can store any value of sales caputured as opposed to fixed values
WITH Annual_sales AS (
  SELECT
    year(Orderdate) AS Year_,
    round(SUM(totalsales),2) AS total_revenue
  FROM sales
  GROUP BY Year_
)
SELECT --result set
  Year_,
  round(total_revenue,2) as Annual_Revenue, 
  LAG(total_revenue) OVER (ORDER BY year_) AS Prev_year_annual_Revenue,
  ROUND((total_revenue - LAG(total_revenue) OVER (ORDER BY year_)) / LAG(total_revenue) OVER (ORDER BY year_) * 100,2) AS Year_on_Year_growth_Percentage
FROM annual_sales;


---Q 3.5 Regional Performance Ranking
select Region, Region_Total_Sales, Region_Total_Orders, Region_rank
from
( select
  customers.region, 
  round(sum(sales.totalsales),2) as Region_Total_Sales,
  count(sales.orderid) as Region_Total_Orders,
row_number() over( order by sum(sales.totalsales) desc) as Region_rank 
from sales
inner join customers
on customers.customerid = sales.customerid
group by customers.region
);

------------------------------------------------------------------------------------------------
--Part 4: Business Intelligence Questions

--Q 4.1 Customer Satisfaction vs. Repeat Purchases

--multi-level aggregation? conversion of data types, string to int, count to avg?
select count(customer_Feedback.orderid) as Total_number_of_orders_per_customer, customer_feedback.Rating,
case 
when Rating between 1 and 3 then "Less satisfied customers"
else "Highly satisfied customers"
end as Rating_Group
from customer_feedback
group by  customer_feedback.rating 
order by customer_feedback.rating desc;

--O4.2 Discount Effectiveness
select Discountpercent, count(orderid) as Number_of_orders_per_discount_band, round(sum(totalsales),2) as Total_sales, round(sum(profit),2) as Total_profit, ROUND(100* Total_Profit/SUM(totalsales), 2) as Profit_Margin_Percent, 
case  
when discountpercent between 1 and 10 then "Low Discount" -- dont use operators, it doesnt work <=
when discountpercent between 11 and 20 then "Mid Discount"
when discountpercent between 21 and 30 then "High Discount"
else "Zero Discount"
end as Discount_Band
from sales
group by DiscountPercent
order by profit_margin_percent;

--Q4.3 Product Portfolio Optimization
WITH ProductOptimization AS (
    SELECT 
      round(SUM(sales.totalsales),2) AS Sales_Volume,
      round(sum(sales.profit),2) AS Total_Profit,
      avg(customer_feedback.rating) as Rating_Ave,
      sales.productid as Product_ID,
      products.productname   
    FROM Products  --fk table
    LEFT JOIN sales  ON sales.ProductID = Products.productid
    LEFT JOIN customer_feedback ON sales.OrderID = customer_feedback.OrderID
    GROUP BY rating, sales.productid, products.ProductName
)
SELECT * 
FROM ProductOptimization
WHERE Sales_Volume < (SELECT AVG(Sales_Volume) FROM ProductOptimization) -- Below average sales
  AND Rating_Ave < 3  -- Low customer satisfaction
ORDER BY Sales_Volume ASC, Rating_Ave ASC
limit 5;

--Part 5: Executive Management Report
--Q5.1 Overall Business Performance- a summary of key business metrics:
select year(orderdate) as Year_, round(sum(totalsales),2) as Total_revenue, round(sum(profit),2) as Total_profit, round(100* Total_profit/SUM(totalsales), 2) as Profit_Margin_Percent_per_Year, round(avg(totalsales),2) as Average_order_value, count(orderid) as Total_num_of_orders, count(CustomerID)as Total_number_of_Customers
from sales
group by Year_;

--Q5.2
--Insights #1: Nova Retail Group has two areas that have improved for the financial year (FY) 2024, namely the Total revenue has increased as --well as a higher total number of orders compared to 2023. However the profit margin for 2024 dipped slightly compared to 2023, as reflected --in the result set below.

--Supporting SQL Query:
select year(orderdate) as Year_, round(sum(totalsales),2) as Total_revenue, round(sum(profit),2) as Total_profit, round(100* Total_profit/SUM(totalsales), 2) as Profit_Margin_Percent_per_Year, round(avg(totalsales),2) as Average_order_value, count(orderid) as Total_num_of_orders, count(CustomerID)as Total_number_of_Customers
from sales
group by Year_;

--Insight #2: Based on the above insight to further give us enlightenment  as to what could be the reason that led to greater revenue in 2024,
--it becomes necessary to examine other areas of the business.
--e-commerce is on rise this can be seen by the higher number of orders placed via the online channel in 2024 compared to FY 2023, these --results serve as proof, consequently the online channel contributed more to the total sales compared to in-store sales for both 2024 and --2023 respectively.

--Supporting SQL Query:
select year(orderdate) as Year_, Channel, count(orderid) as Number_of_Orders, round(avg(totalsales),2) as Average_Order_value, round(sum(totalsales),2) as Total_Sales_Channel
from sales
group by year_, channel;

--Insight #3: Another possibility that could explain the increase in revenue for FY 2024 is by analysing if there was an improvement in the --ratings per product category for each year, the result set for the below SQL query, does not show significant shifts for product category --ratings, however the number of reviews increased meaning customers submitted more reviews but were not generally enthusiastic about the --products as can be seen by the averages of the ratings that remained relatively the same for both 2023 and 2024 respectively.

--Supporting SQL Query:
select year(feedbackdate) as Year_, ProductCategory, round(avg(rating),2) as Average_Rating_per_Category, count(Satisfaction) as Number_of_Reviews 
from customer_feedback
group by year_, ProductCategory
having Number_of_Reviews >50;



-- Section 3: Strategic Recommendations

--Analysing the comments from the Customer_feedback table from face value and from result set of the below SQL query, a majority of --customers are very pleased with the product and service offering from Nova Retail Group, however focus must also be given to the Less --satisfied customers in order to retain them and improve thier user experience to generate even more sales, perhaps management can consider --making contact with the customers to understand their concerns and address them effectively in addition to the following recommendations.

--Actionable recommendations based on the comments(data) from the Customer_feedback table:

--Analysing the comments from the Customer_feedback table from face value and from result set of the below SQL query, a majority of --customers are very pleased with the product and service offering from Nova Retail Group, however focus must also be given to the Less --satisfied customers in order to retain them and improve their user experience to generate even more sales, perhaps management can consider --making contact with the customers to understand their concerns and address them effectively in addition to the following recommendations.


--Actionable recommendations based on the comments(data) from the Customer_feedback table:


--1. Improve the quality of the products in each category. Management can conduct research in order to find suppliers who offer better -- -- --quality products. See Feedbackid FB00123 and FB00411.


--2. Improve delivery times and delivery methods. Regarding the delivery times, Management can ensure to allocate adequate man-power and --resources for the number of orders that must be processed, in addition find quicker delivery routes in the region and or consider collection --hubs for outlier areas in a region, customers may be willing to collect orders themselves as long as their order is delivered according to --the ETA issued by the company. See Feedbackid FB00159.


--3. Ensure careful product handling during shipment to avoid damages. Management needs to find out at what point in the supply chain do --products get damaged, if for example damages occur in the delivery truck then measures should be taken to avoid/minimise in-transit damages, -- management can also find out if given the nature of the product e.g. liquids, glass or electronics are those products wrapped with suitable --packaging materials. See Feedbackid FB00444 and FB00320.


--4. Provide clearer product descriptions. Management can request for the suppliers to provide comprehensive product specifications and
--descriptions along with photographs and or videos. See Feedbackid FB00573.


--5. Improve the customer service experience both online and in-store. Management can provide customer service training to their staff members --to ensure standard customer service on all channels from placing an order to the delivery point - the service must be excellent. See --Feedbackid FB00525.


--Supporting SQL Query:
select *
from customer_feedback;

select count(customer_Feedback.orderid) as Total_number_of_orders_per_customer, customer_feedback.Rating,
case 
when Rating between 1 and 3 then "Less satisfied customers"
else "Highly satisfied customers"
end as Rating_Group
from customer_feedback
group by  customer_feedback.rating 
order by customer_feedback.rating desc;





